# KrushikaDhara ONNX to TFLite Conversion & Validation

**Instructions:**
1. Upload `best_model.onnx`
2. Upload `best_model.pt`
3. Upload `dataset_split/test/` (Upload as a zip and unzip it here)
4. Upload `disease_labels.txt` (to map class indices correctly)

In [ ]:
!pip install -U onnx onnxruntime tensorflow onnx2tf onnxsim torch torchvision torchaudio pillow numpy

In [ ]:
import sys
import onnx
import tensorflow as tf
import numpy as np
import glob
import os
import json
import torch
import onnxruntime as ort
from PIL import Image

print("===============================")
print("Environment Versions")
print(f"Python Version: {sys.version}")
print(f"TensorFlow Version: {tf.__version__}")
print(f"ONNX Version: {onnx.__version__}")
print("===============================")

In [ ]:
onnx_path = "best_model.onnx"
pt_path = "best_model.pt"
tf_model_path = "saved_model"
tflite_path = "crop_disease_classifier_int8.tflite"
test_dir = "test/"
labels_path = "disease_labels.txt"

if not os.path.exists(onnx_path):
    raise FileNotFoundError(f"{onnx_path} missing")
if not os.path.exists(pt_path):
    raise FileNotFoundError(f"{pt_path} missing")
if not os.path.exists(test_dir):
    raise FileNotFoundError(f"{test_dir} missing (unzip test dataset first)")

with open(labels_path, "r") as f:
    labels = [l.strip() for l in f.readlines() if l.strip()]
if len(labels) != 25:
    raise ValueError(f"Expected 25 labels, got {len(labels)}")

In [ ]:
print("Verifying ONNX model...")
onnx_model = onnx.load(onnx_path)

input_shape = []
for inp in onnx_model.graph.input:
    input_shape = [dim.dim_value for dim in inp.type.tensor_type.shape.dim]
    print(f"ONNX Input: {inp.name}, Shape: {input_shape}, DType: {inp.type.tensor_type.elem_type}")

output_shape = []
for out in onnx_model.graph.output:
    output_shape = [dim.dim_value for dim in out.type.tensor_type.shape.dim]
    print(f"ONNX Output: {out.name}, Shape: {output_shape}, DType: {out.type.tensor_type.elem_type}")

if output_shape[-1] != 25:
    raise ValueError(f"ONNX model does not have 25 output classes! Got {output_shape[-1]}")

In [ ]:
print("Converting ONNX to TensorFlow SavedModel using onnx2tf...")
!onnx2tf -i {onnx_path} -o {tf_model_path}
print("SavedModel export successful.")

In [ ]:
print("Converting to INT8 TFLite with Calibration...")

def representative_dataset():
    image_paths = glob.glob(f"{test_dir}/*/*.*")
    np.random.shuffle(image_paths)
    count = 0
    for img_path in image_paths:
        if count >= 100: break
        try:
            img = Image.open(img_path).convert("RGB")
            img = img.resize((224, 224))
            img_array = np.array(img, dtype=np.float32) / 255.0
            mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
            std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
            img_array = (img_array - mean) / std
            img_array = np.transpose(img_array, (2, 0, 1)) # NCHW
            img_array = np.expand_dims(img_array, axis=0)
            count += 1
            yield [img_array]
        except Exception as e:
            print(f"Skipping image due to error: {e}")

converter = tf.lite.TFLiteConverter.from_saved_model(tf_model_path)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.int8
converter.inference_output_type = tf.int8

tflite_model = converter.convert()
with open(tflite_path, "wb") as f:
    f.write(tflite_model)
print("TFLite INT8 conversion successful!")

In [ ]:
print("TFLite Metadata Verification...")
interpreter = tf.lite.Interpreter(model_path=tflite_path)
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()[0]
output_details = interpreter.get_output_details()[0]

if output_details["shape"][-1] != 25:
    raise ValueError(f"TFLite model does not have 25 output classes! Got {output_details['shape'][-1]}")

metadata = {
    "input_shape": input_details["shape"].tolist(),
    "input_dtype": str(input_details["dtype"]),
    "input_scale": float(input_details["quantization"][0]),
    "input_zero_point": int(input_details["quantization"][1]),
    "output_shape": output_details["shape"].tolist(),
    "output_dtype": str(output_details["dtype"]),
    "output_scale": float(output_details["quantization"][0]),
    "output_zero_point": int(output_details["quantization"][1])
}

print(json.dumps(metadata, indent=4))
with open("tflite_metadata.json", "w") as f:
    json.dump(metadata, f, indent=4)

In [ ]:
print("Running REAL inference on 100 test images for validation...")

pt_model = torch.jit.load(pt_path, map_location=torch.device('cpu'))
pt_model.eval()
ort_session = ort.InferenceSession(onnx_path)

def preprocess_image(image_path, format="nchw", dtype=np.float32):
    img = Image.open(image_path).convert("RGB")
    img = img.resize((224, 224))
    img_array = np.array(img, dtype=np.float32) / 255.0
    mean = np.array([0.485, 0.456, 0.406], dtype=np.float32)
    std = np.array([0.229, 0.224, 0.225], dtype=np.float32)
    img_array = (img_array - mean) / std
    if format == "nchw":
        img_array = np.transpose(img_array, (2, 0, 1))
    img_array = np.expand_dims(img_array, axis=0).astype(dtype)
    return img_array

def softmax(x):
    e_x = np.exp(x - np.max(x))
    return e_x / e_x.sum(axis=1, keepdims=True)

test_images = glob.glob(f"{test_dir}/*/*.*")
np.random.shuffle(test_images)
test_images = test_images[:100]

in_scale, in_zero = metadata["input_scale"], metadata["input_zero_point"]
out_scale, out_zero = metadata["output_scale"], metadata["output_zero_point"]
tflite_input_format = "nhwc" if metadata["input_shape"] == [1, 224, 224, 3] else "nchw"

results = []
pt_correct = 0
ort_correct = 0
tf_correct = 0
pt_onnx_agree = 0
pt_tf_agree = 0
onnx_tf_agree = 0

for img_path in test_images:
    true_class = os.path.basename(os.path.dirname(img_path))
    
    pt_in = preprocess_image(img_path, format="nchw", dtype=np.float32)
    tf_in_float = preprocess_image(img_path, format=tflite_input_format, dtype=np.float32)
    tf_in_quant = np.clip(np.round(tf_in_float / in_scale + in_zero), -128, 127).astype(np.int8)
    
    # PyTorch
    with torch.no_grad():
        pt_out = pt_model(torch.from_numpy(pt_in)).numpy()
    pt_pred = int(np.argmax(pt_out[0]))
    
    # ONNX
    ort_inputs = {ort_session.get_inputs()[0].name: pt_in}
    ort_out = ort_session.run(None, ort_inputs)[0]
    ort_pred = int(np.argmax(ort_out[0]))
    
    # TFLite
    interpreter.set_tensor(input_details['index'], tf_in_quant)
    interpreter.invoke()
    tf_out_quant = interpreter.get_tensor(output_details['index'])[0]
    tf_out_float = (tf_out_quant.astype(np.float32) - out_zero) * out_scale
    tf_pred = int(np.argmax(tf_out_float))
    
    true_idx = labels.index(true_class) if true_class in labels else -1
    
    if pt_pred == true_idx: pt_correct += 1
    if ort_pred == true_idx: ort_correct += 1
    if tf_pred == true_idx: tf_correct += 1
    
    if pt_pred == ort_pred: pt_onnx_agree += 1
    if pt_pred == tf_pred: pt_tf_agree += 1
    if ort_pred == tf_pred: onnx_tf_agree += 1
    
    results.append({
        "image": img_path,
        "true": true_class,
        "pt_pred": labels[pt_pred],
        "ort_pred": labels[ort_pred],
        "tf_pred": labels[tf_pred]
    })

n = len(results)
conversion_report = {
    "accuracy": {
        "pytorch": pt_correct / n,
        "onnx": ort_correct / n,
        "tflite": tf_correct / n
    },
    "agreement": {
        "pytorch_vs_onnx": pt_onnx_agree / n,
        "pytorch_vs_tflite": pt_tf_agree / n,
        "onnx_vs_tflite": onnx_tf_agree / n
    }
}

print("\n=== ACCURACY (100 images) ===")
print(f"PyTorch: {conversion_report['accuracy']['pytorch']*100:.2f}%")
print(f"ONNX: {conversion_report['accuracy']['onnx']*100:.2f}%")
print(f"TFLite: {conversion_report['accuracy']['tflite']*100:.2f}%")

print("\n=== AGREEMENT ===")
print(f"PyTorch vs ONNX: {conversion_report['agreement']['pytorch_vs_onnx']*100:.2f}%")
print(f"PyTorch vs TFLite: {conversion_report['agreement']['pytorch_vs_tflite']*100:.2f}%")
print(f"ONNX vs TFLite: {conversion_report['agreement']['onnx_vs_tflite']*100:.2f}%")

with open("conversion_report.json", "w") as f:
    json.dump(conversion_report, f, indent=4)

if conversion_report['agreement']['onnx_vs_tflite'] < 0.90:
    raise RuntimeError("ONNX vs TFLite agreement is unexpectedly poor (<90%).")

print("\nSUCCESS! Conversion and validation passed. Please download:")
print("1. crop_disease_classifier_int8.tflite")
print("2. tflite_metadata.json")
print("3. conversion_report.json")